In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from time import sleep 
from PIL import Image

from skimage.segmentation import active_contour
from skimage.filters import gaussian
from skimage.color import rgb2gray
from skimage import img_as_float

# CONFIGURARE ȘI CALE
cale_imagine = r'/Users/ana/Documents/Lung Cancer Diagnosis System/Imagini pentru analiza/cancere/tu pulm lsd bruma mihaila bio 4.jpg' 

drawing = False 

try:
    img_pil = Image.open(cale_imagine)
    road_bgr = np.array(img_pil)
    road_original = cv2.cvtColor(road_bgr, cv2.COLOR_RGB2BGR)
    
    if road_original is None:
        raise FileNotFoundError(f"Nu s-a putut încărca imaginea de la calea: {cale_imagine}")

    road_gray = cv2.cvtColor(road_original, cv2.COLOR_BGR2GRAY)
    
    #CONTUR ALB 
    clahe_roi = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_np_contrast = clahe_roi.apply(road_gray)
    img_blurred_roi = cv2.GaussianBlur(img_np_contrast, (9, 9), 0)
    
    _, thresh_delimitare = cv2.threshold(img_blurred_roi, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    kernel_morph = np.ones((10, 10), np.uint8) 
    thresh_clean = cv2.morphologyEx(thresh_delimitare, cv2.MORPH_CLOSE, kernel_morph)

    contours_ecografie, _ = cv2.findContours(thresh_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    mask_alb = np.zeros_like(road_gray)
    largest_contour_delimitare = None
    
    if contours_ecografie:
        largest_contour_delimitare = max(contours_ecografie, key=cv2.contourArea)
        cv2.drawContours(mask_alb, [largest_contour_delimitare], -1, 255, -1) 
        print("Conturul Alb a fost calculat.")

    # PRE-PROCESARE PENTRU WATERSHED 
    clahe_ws = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)) 
    road_contrast = clahe_ws.apply(road_gray)
    road_blurred = cv2.GaussianBlur(road_contrast, (7, 7), 0)
    road_bgr_processed = cv2.cvtColor(road_blurred, cv2.COLOR_GRAY2BGR) 

except FileNotFoundError as e:
    print(f"Eroare: {e}. Asigură-te că fisierul există.")
    exit()

# Variabile Globale și de Display
# Imaginea de bază peste care desenăm
road_base = np.copy(road_bgr_processed)
if largest_contour_delimitare is not None:
    cv2.drawContours(road_base, [largest_contour_delimitare], -1, (255, 255, 255), 2)

# road_copy
road_copy = road_base.copy()

marker_image = np.zeros(road_bgr_processed.shape[:2], dtype=np.int32)
segments = np.zeros(road_bgr_processed.shape, dtype=np.uint8)

def create_rgb(i):
    if i == 1:
        return (0, 255, 0)   # Verde (Tumoare)
    elif i == 2:
        return (0, 0, 255)   # Roșu (Țesut sănătos)
    else:
        x = np.array(cm.get_cmap('tab10')(i))[:3] * 255
        return tuple(x.astype(int).tolist())

colors = [create_rgb(i) for i in range(10)]

# Global Variables
current_marker = 1 
marks_updated = False       
snake_contour = None
last_segments = segments.copy()  # Ultimul rezultat Watershed

# CALLBACK FUNCTION
def mouse_callback(event, x, y, flags, param):
    global marks_updated, road_copy, current_marker, drawing
    
    # Părți mai mici din tumoare / tesut sănătos
    draw_radius = 5  

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        if mask_alb[y, x] == 255:
            marks_updated = True
            cv2.circle(marker_image, (x, y), draw_radius, (current_marker), -1) 
            cv2.circle(road_copy, (x, y), draw_radius, colors[current_marker], -1)

    elif event == cv2.EVENT_MOUSEMOVE and drawing:
        if mask_alb[y, x] == 255:
            marks_updated = True
            cv2.circle(marker_image, (x, y), draw_radius, (current_marker), -1) 
            cv2.circle(road_copy, (x, y), draw_radius, colors[current_marker], -1)
        
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False


cv2.namedWindow('Imagine Ecografica (Watershed)', cv2.WINDOW_NORMAL)
cv2.setMouseCallback('Imagine Ecografica (Watershed)', mouse_callback)

while True:
    display_image = road_copy.copy() 

    # Desenăm conturul Snake, dacă există
    if snake_contour is not None:
        snake_coords = np.array(snake_contour).astype(np.int32)
        snake_coords_xy = np.fliplr(snake_coords) 
        
        cv2.polylines(display_image, [snake_coords_xy], isClosed=True, color=(0, 0, 255), thickness=2) 

    cv2.imshow('Segmente Watershed', last_segments)
    cv2.imshow('Imagine Ecografica (Watershed)', display_image) 

    k = cv2.waitKey(1)
    
    if k == 27: # ESC
        break

    elif k == ord('c'): # Clear
        print("Curăț toate marcajele și segmentele.")
        marker_image = np.zeros(road_bgr_processed.shape[:2], dtype=np.int32)
        last_segments = np.zeros(road_bgr_processed.shape, dtype=np.uint8)
        snake_contour = None
        marks_updated = False
        road_copy = road_base.copy()

    elif k >= ord('0') and k <= ord('9'): 
        current_marker = int(chr(k))
        print(f"Culoare selectată: {current_marker} (1=Tumoare/Verde, 2=Țesut sănătos/Roșu)")

    #WATERSHED
    elif k == ord('w'):
        print("Rulăm Watershed")

        if not np.any(marker_image == 1):
            print(" Nu există markeri de tumoare (1). Desenează întâi cu tasta '1'.")
            continue
        if not np.any(marker_image == 2):
            print("Nu există markeri de țesut sănătos (2). Desenează întâi cu tasta '2'.")
            continue

        marker_image_copy = marker_image.copy()
        
        cv2.watershed(road_bgr_processed, marker_image_copy)
        segments = np.zeros(road_bgr_processed.shape, dtype=np.uint8)

        for color_ind in range(10):
            segments[marker_image_copy == (color_ind)] = colors[color_ind]

        segments = cv2.bitwise_and(segments, segments, mask=mask_alb)
        
        segments_gray = cv2.cvtColor(segments, cv2.COLOR_BGR2GRAY)
        _, segments_mask = cv2.threshold(segments_gray, 1, 255, cv2.THRESH_BINARY)

        segments_output = cv2.bitwise_and(segments, segments, mask=segments_mask)
        
        road_blend = cv2.addWeighted(road_bgr_processed, 0.9, segments_output, 0.4, 0)  
        
        road_copy = road_blend.copy()
        if largest_contour_delimitare is not None:
            cv2.drawContours(road_copy, [largest_contour_delimitare], -1, (255, 255, 255), 2)
        
        last_segments = segments.copy() # Salvăm segmentele Watershed pentru Snake
        snake_contour = None 
        marks_updated = False

        print("Watershed finalizat.")

    # SNAKE
    elif k == ord('s'):
        print("Rulăm Active Contour")
        
        tumora_mask_bgr = last_segments.copy()
        tumora_mask = np.zeros_like(road_gray, dtype=np.uint8)
        tumora_mask[(tumora_mask_bgr[:,:,0] == 0) & (tumora_mask_bgr[:,:,1] == 255) & (tumora_mask_bgr[:,:,2] == 0)] = 255
        
        contours_tumora_watershed, _ = cv2.findContours(tumora_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if contours_tumora_watershed:
            initial_snake_contour = max(contours_tumora_watershed, key=cv2.contourArea)
            
            initial_snake_coords = initial_snake_contour.squeeze().astype(float)
            initial_snake_coords = np.fliplr(initial_snake_coords) 
            
            img_for_snake = gaussian(img_as_float(road_gray), sigma=1.0, preserve_range=False)
            
            # PARAMETRI SNAKE 
            alpha_val = 0.01 
            beta_val = 0.01  
            gamma_val = 0.001
            w_edge_val = 1000.0 
            w_line_val = -100.0 

            try:
                snake_contour = active_contour(img_for_snake,
                                               initial_snake_coords,
                                               alpha=alpha_val, beta=beta_val, gamma=gamma_val,
                                               w_edge=w_edge_val, w_line=w_line_val 
                                              )
                print(f"Snake rulat cu succes.")
                print(f"Parametri finali: alpha={alpha_val}, beta={beta_val}, w_edge={w_edge_val}, w_line={w_line_val}")
            except Exception as snake_e:
                print(f"Eroare la rularea Snake: {snake_e}")
                snake_contour = None
        else:
            print("Nu s-a găsit niciun contur Verde din Watershed pentru a inițializa Snake.")

cv2.destroyAllWindows()

Conturul Alb a fost calculat.


/var/folders/mm/w5mb7krj1vb1hrlfwfq3439h0000gn/T/ipykernel_33898/2227337030.py:74: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  x = np.array(cm.get_cmap('tab10')(i))[:3] * 255
2026-02-12 22:45:30.837 Python[33898:4186369] +[IMKClient subclass]: chose IMKClient_Modern
2026-02-12 22:45:30.837 Python[33898:4186369] +[IMKInputSession subclass]: chose IMKInputSession_Modern


KeyboardInterrupt: 